# 🚀 Detector Deepfake Definitivo (RTX 5060 / Blackwell)

Este cuaderno está preparado para entrenar en tu RTX 5060 evitando errores de compatibilidad CUDA.

## Orden obligatorio
1. Ejecuta la celda 1 (instalación).
2. Reinicia el kernel.
3. Ejecuta las celdas 2 a 6 en orden.

# 1) Instalación limpia de dependencias (Nightly cu128)

Instala exactamente lo necesario para entrenar imágenes con RTX 5060 Blackwell (sm_120).

Al terminar, reinicia el kernel antes de seguir.

In [1]:
import subprocess
import sys

INDEX_URL = "https://download.pytorch.org/whl/nightly/cu128"


def run_pip(args, allow_fail=False):
    cmd = [sys.executable, "-m", "pip"] + args
    print(f"$ {' '.join(cmd)}")
    result = subprocess.run(cmd, text=True)
    if result.returncode != 0 and not allow_fail:
        raise RuntimeError(f"Falló comando pip: {' '.join(args)}")
    return result.returncode == 0


print("=" * 70)
print("🔧 INSTALACIÓN DEFINITIVA PARA RTX 5060 (BLACKWELL / sm_120)")
print("=" * 70)
print(f"🐍 Python del kernel: {sys.executable}")
print(f"📦 Índice Nightly: {INDEX_URL}")

run_pip(["install", "-U", "pip"])
run_pip(["uninstall", "-y", "torch", "torchvision", "torchaudio"], allow_fail=True)

# Stack compatible actual para Blackwell
run_pip(["install", "--upgrade", "--pre", "torch", "torchvision", "--index-url", INDEX_URL])

# Utilidades de entrenamiento
run_pip(["install", "tensorboard", "tqdm", "pillow", "pandas", "matplotlib"])

# Verificación real de instalación
try:
    import torch  # noqa: F401
    import torchvision  # noqa: F401
    print(f"✅ torch OK: {torch.__version__}")
    print(f"✅ torchvision OK: {torchvision.__version__}")
except Exception as e:
    raise RuntimeError(f"Instalación incompleta. Detalle: {e}")

print("\n✅ Instalación completada correctamente.")
print("⚠️ Reinicia el kernel ahora y continúa con la celda 2.")

🔧 INSTALACIÓN DEFINITIVA PARA RTX 5060 (BLACKWELL / sm_120)
🐍 Python del kernel: d:\proyectos\Detector de ia\.venv312\Scripts\python.exe
📦 Índice Nightly: https://download.pytorch.org/whl/nightly/cu128
$ d:\proyectos\Detector de ia\.venv312\Scripts\python.exe -m pip install -U pip
$ d:\proyectos\Detector de ia\.venv312\Scripts\python.exe -m pip uninstall -y torch torchvision torchaudio
$ d:\proyectos\Detector de ia\.venv312\Scripts\python.exe -m pip install --upgrade --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu128
$ d:\proyectos\Detector de ia\.venv312\Scripts\python.exe -m pip install tensorboard tqdm pillow pandas matplotlib
✅ torch OK: 2.12.0.dev20260228+cu128
✅ torchvision OK: 0.26.0.dev20260221+cu128

✅ Instalación completada correctamente.
⚠️ Reinicia el kernel ahora y continúa con la celda 2.


# 2) Verificación real de CUDA + preparación de logs

Esta celda confirma que la instalación sí puede ejecutar kernels CUDA en tu GPU antes de entrenar.

In [3]:
import os
import sys
import logging
import subprocess
from datetime import datetime

INDEX_URL = "https://download.pytorch.org/whl/nightly/cu128"
EXPECTED_ENV_HINT = os.path.join("Detector de ia", ".venv312").lower().replace("\\", "/")


def ensure_torch_stack():
    try:
        import torch
        import torchvision
        return torch, torchvision
    except Exception:
        print("⚠️ torch/torchvision no disponibles en este kernel. Instalando ahora...")
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "--upgrade", "--pre", "torch", "torchvision", "--index-url", INDEX_URL
        ])
        import torch
        import torchvision
        print("✅ Instalación en caliente completada. Si persiste algo, reinicia kernel una vez.")
        return torch, torchvision


torch, torchvision = ensure_torch_stack()
from torch.utils.tensorboard import SummaryWriter

print("=" * 70)
print("🔍 VERIFICACIÓN DE ENTORNO")
print("=" * 70)
kernel_python = sys.executable.lower().replace("\\", "/")
print(f"🐍 Python del kernel: {sys.executable}")

if EXPECTED_ENV_HINT not in kernel_python:
    raise RuntimeError(
        "Kernel incorrecto detectado. Selecciona el intérprete .venv312 del proyecto y reinicia kernel."
    )

print(f"📦 Torch: {torch.__version__}")
print(f"📦 Torchvision: {torchvision.__version__}")
print(f"🧩 CUDA compilada en Torch: {torch.version.cuda}")
print(f"⚡ torch.cuda.is_available(): {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA no está disponible en este kernel. "
        "Ejecuta la celda 1, reinicia kernel y selecciona el entorno correcto."
    )

device = torch.device("cuda")
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3

# Ajustes de rendimiento para convoluciones
torch.backends.cudnn.benchmark = True
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print(f"✅ GPU detectada: {gpu_name}")
print(f"✅ VRAM: {vram_gb:.2f} GB")

# Test real para detectar incompatibilidades de kernel CUDA
try:
    x = torch.randn(2, 3, 64, 64, device=device)
    conv = torch.nn.Conv2d(3, 8, 3, padding=1).to(device)
    y = conv(x)
    _ = y.mean().item()
    torch.cuda.synchronize()
    print("✅ Test de kernel CUDA superado (compatibilidad OK)")
except Exception as e:
    raise RuntimeError(
        "Fallo de compatibilidad CUDA en ejecución real. "
        "Reejecuta celda 1 (cu128), reinicia kernel y vuelve a esta celda.\n"
        f"Detalle: {e}"
    )

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("DeepfakeRTX5060")

os.makedirs("checkpoints", exist_ok=True)
os.makedirs("runs", exist_ok=True)

run_id = datetime.now().strftime("%Y%m%d-%H%M%S")
writer = SummaryWriter(f"runs/deepfake_rtx5060_{run_id}")
print(f"⭐ Experimento: deepfake_rtx5060_{run_id}")

🔍 VERIFICACIÓN DE ENTORNO
🐍 Python del kernel: d:\proyectos\Detector de ia\.venv312\Scripts\python.exe
📦 Torch: 2.12.0.dev20260228+cu128
📦 Torchvision: 0.26.0.dev20260221+cu128
🧩 CUDA compilada en Torch: 12.8
⚡ torch.cuda.is_available(): True
✅ GPU detectada: NVIDIA GeForce RTX 5060 Laptop GPU
✅ VRAM: 7.96 GB
✅ Test de kernel CUDA superado (compatibilidad OK)
⭐ Experimento: deepfake_rtx5060_20260228-115417


# 3) Carga de dataset y DataLoaders (RAM-safe Windows)

Configuración optimizada para evitar saturación de RAM del sistema en Windows.
Requiere estructura:
- dataset_deepdetect/ddata/train/{fake,real}
- dataset_deepdetect/ddata/test/{fake,real}

In [4]:
import os
import multiprocessing
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

DATASET_ROOT = os.path.join("dataset_deepdetect", "ddata")
TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "test")

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(VAL_DIR):
    raise FileNotFoundError(
        f"No se encontró el dataset esperado.\nTrain: {TRAIN_DIR}\nVal: {VAL_DIR}"
    )

# Reducir presión de CPU/RAM en pipeline de imagen
RAM_SAFE_MODE = True

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

val_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_transforms)
class_names = train_dataset.classes

if RAM_SAFE_MODE:
    BATCH_SIZE = 16
else:
    BATCH_SIZE = 24

if os.name == "nt":
    NUM_WORKERS = 0
    persistent_workers = False
else:
    NUM_WORKERS = max(0, min(4, multiprocessing.cpu_count() - 2))
    persistent_workers = NUM_WORKERS > 0

# En modo RAM-safe, desactivar pin_memory reduce presión de RAM host
PIN_MEMORY = (device.type == "cuda") and (not RAM_SAFE_MODE)

loader_kwargs = {
    "pin_memory": PIN_MEMORY,
    "num_workers": NUM_WORKERS,
}
if NUM_WORKERS > 0:
    loader_kwargs["persistent_workers"] = persistent_workers

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

print("✅ Dataset cargado")
print(f"   Clases: {class_names}")
print(f"   Train: {len(train_dataset)} imágenes")
print(f"   Val: {len(val_dataset)} imágenes")
print(f"   Batch size: {BATCH_SIZE} | Workers: {NUM_WORKERS} | pin_memory: {PIN_MEMORY}")
print(f"   RAM_SAFE_MODE: {RAM_SAFE_MODE}")

✅ Dataset cargado
   Clases: ['fake', 'real']
   Train: 90409 imágenes
   Val: 21776 imágenes
   Batch size: 16 | Workers: 0 | pin_memory: False
   RAM_SAFE_MODE: True


# 4) Modelo ConvNeXt Tiny (transfer learning)

In [5]:
import torch
import torch.nn as nn
from torchvision import models

class DeepfakeDetector(nn.Module):
    def __init__(self, num_classes=2, freeze_backbone=True):
        super().__init__()
        self.model = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)

        if freeze_backbone:
            for p in self.model.parameters():
                p.requires_grad = False

        in_features = self.model.classifier[2].in_features
        self.model.classifier[2] = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(in_features, num_classes),
        )

    def forward(self, x):
        return self.model(x)

model = DeepfakeDetector(num_classes=len(class_names), freeze_backbone=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.1, patience=2)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ Modelo listo | Parámetros entrenables: {trainable_params:,}")

✅ Modelo listo | Parámetros entrenables: 1,538


# 5) Entrenamiento estable (AMP + mejor modelo)

Entrena con mixed precision y guarda automáticamente el mejor checkpoint en checkpoints/model_best.pth.

In [6]:
from tqdm import tqdm
import torch
import gc

EPOCHS = 3
BEST_ACC = 0.0

use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)


def run_epoch(loader, train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    bar = tqdm(loader, desc="Train" if train else "Val", leave=False, mininterval=1.0)
    for step, (images, labels) in enumerate(bar, start=1):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast("cuda", enabled=use_amp):
                outputs = model(images)
                loss = criterion(outputs, labels)

            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        total_correct += (preds == labels).sum().item()
        total_samples += labels.size(0)

        if step % 20 == 0 or step == len(loader):
            bar.set_postfix({
                "loss": f"{total_loss / total_samples:.4f}",
                "acc": f"{100 * total_correct / total_samples:.2f}%"
            })

        del images, labels, outputs, loss, preds
        if step % 200 == 0:
            gc.collect()

    return total_loss / total_samples, 100 * total_correct / total_samples


def train_model():
    global BEST_ACC

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = run_epoch(train_loader, train=True)
        with torch.no_grad():
            val_loss, val_acc = run_epoch(val_loader, train=False)

        writer.add_scalar("Loss/Train", train_loss, epoch)
        writer.add_scalar("Loss/Val", val_loss, epoch)
        writer.add_scalar("Acc/Train", train_acc, epoch)
        writer.add_scalar("Acc/Val", val_acc, epoch)

        scheduler.step(val_acc)

        if val_acc > BEST_ACC:
            BEST_ACC = val_acc
            torch.save({
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_acc": BEST_ACC,
                "class_names": class_names,
            }, "checkpoints/model_best.pth")
            mark = "🌟"
        else:
            mark = "-"

        print(
            f"{mark} Epoch {epoch}/{EPOCHS} | "
            f"train_loss={train_loss:.4f} train_acc={train_acc:.2f}% | "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.2f}% | best={BEST_ACC:.2f}%"
        )

        gc.collect()
        if device.type == "cuda":
            torch.cuda.empty_cache()

    writer.close()
    print(f"\n✅ Entrenamiento completado. Mejor val_acc: {BEST_ACC:.2f}%")
    print("💾 Modelo guardado en: checkpoints/model_best.pth")


print(f"✅ Sistema listo | AMP activo: {use_amp}")

✅ Sistema listo | AMP activo: True


# 6) Iniciar entrenamiento

Ejecuta esta celda para comenzar.

In [ ]:
train_model()

🌟 Epoch 1/3 | train_loss=0.4989 train_acc=76.22% | val_loss=0.9821 val_acc=49.87% | best=49.87%


🌟 Epoch 2/3 | train_loss=0.4264 train_acc=80.64% | val_loss=0.9955 val_acc=51.89% | best=51.89%


🌟 Epoch 3/3 | train_loss=0.4146 train_acc=81.19% | val_loss=0.9787 val_acc=52.65% | best=52.65%

✅ Entrenamiento completado. Mejor val_acc: 52.65%
💾 Modelo guardado en: checkpoints/model_best.pth


: 